# Group original spectrograms
The challenge is that the original spectrogm is too big and overwhelm the working memory (128G), need to think through what to group first, and select those from subject specific data, also need to crop the data.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 15})
import scipy.stats
import os
import mat73
import mne
from mne.time_frequency import tfr_morlet
import pandas as pd
import scipy.io as io
from numpy import loadtxt
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
import sklearn.metrics.pairwise as pairwise
from mpl_toolkits.axes_grid1 import make_axes_locatable
import glob
from joblib import Parallel, delayed

In [2]:
import sys
sys.path.append('/data/dian/Dropbox/scripts/Stanford/ThalamocoricalLoop-project/utils') 
import customFunctions
import importlib
importlib.reload(customFunctions)
from customFunctions import *

In [3]:
os.chdir('/data/dian/Dropbox/Stanford_Matters/data/THAL/CCEP/results/explore5_locked')

#### Perparation

In [4]:
metaT = pd.read_csv('table_CCEPnewpipOutput_wholebrain_anatomical_info_activationRedone2.csv')

In [5]:
sblist = metaT.subject.unique()
sblist = np.delete(sblist, np.where(sblist=='S23_196_HL'))
print(sblist.shape)
print(sblist)
#mat = mergeMat(sblist, keys=['filteridx_metaT'], # only group index, output indexing is in the python couting scheme
 #             inputDir = 'UMAP_learn/resample3')

(26,)
['S21_166_TM' 'S21_167_MQ' 'S21_169_BH' 'S21_170_JL' 'S21_171_MM'
 'S21_172_KS' 'S22_176_LB' 'S22_177_JM' 'S22_178_AF' 'S22_181_CB'
 'S22_182_DH' 'S22_183_CR' 'S22_185_TW' 'S22_188_CB' 'S22_189_LMA'
 'S22_190_AS' 'S22_192_LG' 'S22_193_AM' 'S23_194_PS' 'S23_195_MZ'
 'S23_197_TA' 'S23_198_JP' 'S23_199_GB' 'S23_201_JG' 'S23_202_KC'
 'S23_205_LLC']


In [6]:
brainInfo = pd.read_csv('UMAP/ALLDATA_semisupervise/brainInfo.csv')
isnoise = loadtxt('UMAP/ALLDATA_semisupervise/isnoise.txt',
                delimiter="\t")
cleanFeatures = loadtxt('UMAP/ALLDATA_semisupervise/cleanFeatures.txt',
                delimiter="\t",dtype='str')

In [7]:
hemis = list(map(detHemi,
                 metaT.MNIout_coord_1.tolist(), metaT.MNIin_coord_1.tolist()
          ))
df = pd.DataFrame({'anatomy_conn': list(map(lambda x,y:'-'.join(str(e) for e in [x,y]),
                  cortexLab(metaT.JP_label_out.tolist()),
                  cortexLab(metaT.JP_label_in.tolist()))),#,
                 # metaT.activated.tolist())),
                   'hemis':hemis
                  })
df['anatomy_from'] = cortexLab(metaT.JP_label_out.tolist())
df['anatomy_to']   = cortexLab(metaT.JP_label_in.tolist())
df['anatomy_fromTHAL'] = thalLab(metaT.JP_label_out.tolist())
df['anatomy_toTHAL'] = thalLab(metaT.JP_label_in.tolist())

crossNet_cat = ['internet','intranet']
df['crossNet_bin'] = list(map(lambda x,y:crossNet_cat[int(x==y)],
                         metaT.Yeo7_out2, metaT.Yeo7_in2))
df['crossNet'] = list(map(lambda x,y:'-'.join(str(e) for e in [x,y]),
                         metaT.Yeo7_out2, metaT.Yeo7_in2))
df['Hemi-crossNet_bin'] = list(map(lambda x,y:'-'.join(str(e) for e in [x,y]),
                  df['hemis'],
                  df['crossNet_bin']
                         ))
df['anatomy_from-hemi'] = df['anatomy_from'].str.cat(df['hemis'], sep='-')

In [8]:
# sort some other exclusion criteria 
'''
1.stimID_exclude  = 'S21_167_MQ RE1-RE2',
2. brainInfo-> T.index (index_python back to metaT)
3. isnoise = 0
4. within features of interest
'''
# sort activation labelling - prior information
act = io.loadmat('umapAct.mat') # corresponding to metaT, 0 = inactive, 1 = active, nan = noise
idx_in_metaT_1 =  np.where(act['umapAct'][:,0]==1)[0]
idx_in_metaT_0 = np.where(act['umapAct'][:,0]==0)[0]
boolean_exclude = (((brainInfo['subject']=='S21_167_MQ')& (brainInfo['stim_chan']=='RE1-RE2'))
                    |(isnoise==1)) # n=3892

In [9]:
fti = io.loadmat('spectCCEP/freq_time_info_spectCCEP.mat')

In [10]:
def plot_spectCCEP(d,ax,colmap='inferno',fontsize=12, title_text='', shrink=0.6):
    
    im = ax.imshow(d, aspect=2.1, cmap=colmap, origin='lower')
    
    nsteps = 5
    xticks = np.floor(np.linspace(0, mat['time'].shape[0]-1, nsteps)).astype(int)
    yticks = np.floor(np.linspace(0, mat['freqs'].shape[0]-1, nsteps)).astype(int)
    ax.set_xticks(xticks)
    ax.set_xticklabels(np.rint(1000*mat['time'][xticks]).astype(int))
    ax.set_yticks(yticks)
    ax.set_yticklabels(np.rint(mat['freqs'][yticks]).astype(int))
    
    ax.set_xlabel('time (ms)')
    ax.set_ylabel('frequency (Hz)')
    ax.title.set_text(title_text)
    cbar = plt.colorbar(im, shrink=shrink)
    cbar.ax.tick_params(labelsize=12)
    # add stimulation onset
    zeroPt = np.where(np.abs(mat['time'])==np.min(np.abs(mat['time'])))[0]
    ax.axvline(x=zeroPt, color='w',ls=':', lw=2)
    # adjust font size
    for item in ([ax.title, ax.xaxis.label, ax.yaxis.label] + ax.get_xticklabels() + ax.get_yticklabels()):
        item.set_fontsize(fontsize)
    return ax, im, cbar

def filteridx_of_features(ftype, sblist, active = 1, extraFilterIdx=[]):
    # customized condition in this code
    boolean_include = (cleanFeatures==ftype)
    
    print(sblist)
    if active == 1:
        # filter1
        idx_in_metaT_act = brainInfo['idx_in_metaT'][(~boolean_exclude) & boolean_include].to_numpy()
        idx_in_metaT_include = idx_in_metaT_act
    elif active == 0:
        # filter2 - inactive control to contrast against
        idx_in_metaT_nonact = np.intersect1d(np.setdiff1d(idx_in_metaT_0, 
                                        brainInfo['idx_in_metaT'][(boolean_exclude)] ),
                                  np.where(df['anatomy_from-hemi']==ftype)[0])
        idx_in_metaT_include = idx_in_metaT_nonact
        
    filteridx = np.intersect1d(idx_in_metaT_include, np.where(np.isin(metaT.subject,sblist))[0])
    
    # if use extra filters
    if isinstance(extraFilterIdx, list):
        extraFilterIdx = np.array(extraFilterIdx)
    if extraFilterIdx.shape[0]!=0:
        filteridx = np.intersect1d(filteridx, extraFilterIdx)
    print('Selected %d rows from metaT for %s...'%(filteridx.shape[0], ftype))
        
    return filteridx



def Act_minus_nonact(mat, ftype, key='pw', imputeMissing = 0, toplot=0):
    start_time = time.time()
    # loop through each stimulation channel, to get the activation pattern contracted with the non-activated pattern; 
    #i.e. using each channel's non-activation pattern as control
    allsublist = metaT['subject'][mat['filteridx']].tolist()
    allstimlist = metaT['stim_chan'][mat['filteridx']].tolist()
    allstimID = [i+' '+j for i,j in zip(allsublist, allstimlist)]
    dat = mat[key].reshape(mat[key].shape[0], mat[key].shape[1]*mat[key].shape[2])
    dat_res = dat.copy()
    # predefine non-activated matrix - for plotting
    dat0_all = np.zeros((1, mat[key].shape[1]*mat[key].shape[2]))
    
    for stimID in set(allstimID):
    
        #print('Stim ID: %s'%(stimID))
        stimIdx = set(np.where(np.array(allstimID) == stimID)[0])
        sactIdx = list(stimIdx.intersection(mat['filteridx']))

        # find the corresponding nonactivated data
        sbj = stimID.split(' ')[0]
        stimChan = stimID.split(' ')[1]
        filteridx_stimChan = np.where((metaT['subject']==sbj)&
                                     (metaT['stim_chan']==stimChan))[0]
        filteridx0 = np.intersect1d(filteridx_stimChan, np.setdiff1d(idx_in_metaT_0, 
                                    brainInfo['idx_in_metaT'][(boolean_exclude)] ))
        if filteridx0.shape[0] >= 5:

            mat0 = stackSpectCCEP(filteridx0,
                                  cropdim=[[np.min(mat['time'])-0.001, np.max(mat['time'])+0.001],
                                                      [np.min(mat['freqs'])-1, np.max(mat['freqs'])+1]],
                                  data_type = key,
                                  time_sample=fti['time'], freq_sample=fti['freqs'], 
                                  toplot=0, reporttime=0)

            dat0 = mat0[key].reshape(mat0[key].shape[0], mat0[key].shape[1]*mat0[key].shape[2])
            dat0_0 = dat0.copy()
            dat0_0[np.all(np.isnan(dat0), axis=1)] = 0
            dat_res[sactIdx,:] = dat[sactIdx,:] - np.nanmean(dat0_0, axis = 0) #subtract row vector from matrix
            dat0_all = np.vstack((dat0_all,dat0))
        else:
            print('Not enough (n=%d) non-activated control data to be substracted with, use original data (%s).'%(filteridx0.shape[0], stimID))
     
            
    # extra cleaning step: impute missing data
    if imputeMissing == 1:
        # Preprocess - simulate nan values, which are few
        pipe = make_pipeline(SimpleImputer(strategy="mean"))
        dat_res = pipe.fit_transform(dat_res.copy())
    
    if toplot ==1:
        figdir = '/data/dian/Dropbox/Stanford_Matters/data/THAL/PLOTS/SI'
        # plot to see
        plt.close()
        fig, (ax,ax2) = plt.subplots(1,2,figsize=(15,8))
        plot_spectCCEP(np.nanmean(dat_res.reshape(dat_res.shape[0], -1, mat['time'].shape[0]), axis=0),
                            ax, fontsize=17, title_text = 'cleaned data', shrink=0.4)
        plot_spectCCEP(np.nanmean(dat0_all.reshape(dat0_all.shape[0],-1, mat['time'].shape[0]), axis=0),
                            ax2,  fontsize=17, title_text = 'contrased data', shrink=0.4)
        plt.savefig(figdir+'/GroupAveragedCleanedContrasted_spectra_' + ftype + '_' + key + '.pdf')
        
     # Calculate elapsed time
    end_time = time.time()
    elapsed_time = end_time - start_time
    print("Elapsed time: ", elapsed_time) 
    
    return dat_res
        

## Feature Extraction
There are three important clusters: early peak: represented by the COR-ipsi, ~10-55 ms (beta), late peak: represented by COR-contr, ~75-164 ms (theta), and very late peak: represented by THAL-COR connectivity ~165-310ms (alpha + theta)

In [11]:
ftypes = np.unique(cleanFeatures)
print(ftypes)
t1 = -0.03
t2 = 0.6

['COR-contr' 'COR-ipsi' 'THAL-contr' 'THAL-ipsi']


In [ ]:
for ftype in ftypes:
    print(ftype)
    filteridx = filteridx_of_features(ftype, sblist)
    mat = stackSpectCCEP(filteridx, cropdim=[[t1,t2],[0.5, 125]], data_type = 'pw',
                          time_sample=fti['time'], freq_sample=fti['freqs'], toplot=0)
    dat_res = Act_minus_nonact(mat, ftype, imputeMissing=1, toplot=1, key = 'pw')
    mat['dat_res'] = dat_res
    # save
    io.savemat('groupedSpectPower_%s.mat'%(ftype), mat)

COR-contr
['S21_166_TM' 'S21_167_MQ' 'S21_169_BH' 'S21_170_JL' 'S21_171_MM'
 'S21_172_KS' 'S22_176_LB' 'S22_177_JM' 'S22_178_AF' 'S22_181_CB'
 'S22_182_DH' 'S22_183_CR' 'S22_185_TW' 'S22_188_CB' 'S22_189_LMA'
 'S22_190_AS' 'S22_192_LG' 'S22_193_AM' 'S23_194_PS' 'S23_195_MZ'
 'S23_197_TA' 'S23_198_JP' 'S23_199_GB' 'S23_201_JG' 'S23_202_KC'
 'S23_205_LLC']
Selected 9415 rows from metaT for COR-contr...
Elapsed time:  189.0214352607727
Elapsed time:  3490.655686855316
COR-ipsi
['S21_166_TM' 'S21_167_MQ' 'S21_169_BH' 'S21_170_JL' 'S21_171_MM'
 'S21_172_KS' 'S22_176_LB' 'S22_177_JM' 'S22_178_AF' 'S22_181_CB'
 'S22_182_DH' 'S22_183_CR' 'S22_185_TW' 'S22_188_CB' 'S22_189_LMA'
 'S22_190_AS' 'S22_192_LG' 'S22_193_AM' 'S23_194_PS' 'S23_195_MZ'
 'S23_197_TA' 'S23_198_JP' 'S23_199_GB' 'S23_201_JG' 'S23_202_KC'
 'S23_205_LLC']
Selected 39463 rows from metaT for COR-ipsi...
Elapsed time:  733.4527022838593


In [ ]:
for ftype in ftypes:
    print(ftype)
    filteridx = filteridx_of_features(ftype, sblist)
    mat = stackSpectCCEP(filteridx, cropdim=[[t1,t2],[0.5, 125]], data_type = 'pc',
                          time_sample=fti['time'], freq_sample=fti['freqs'], toplot=0)
    dat_res = Act_minus_nonact(mat, ftype, imputeMissing=1, toplot=1,  key = 'pc')
    mat['dat_res'] = dat_res
    # save
    io.savemat('groupedSpectITPC_%s.mat'%(ftype), mat)

### Save data for other anatomical types

In [ ]:
fromSUBCOR = np.where(metaT['JP_label_out'].isin(['BG','AMY','CLT']))[0]
toSUBCOR = np.where(metaT['JP_label_in'].isin(['BG','AMY','CLT']))[0]

Save for power data:

In [ ]:
##### from SUBCOR mat data
filteridx = np.intersect1d(fromSUBCOR, idx_in_metaT_1)
ftype = 'fromSUBCOR'
mat = stackSpectCCEP(filteridx, cropdim=[[t1,t2],[0.5, 125]], data_type = 'pw',
                          time_sample=fti['time'], freq_sample=fti['freqs'], toplot=0)
dat_res = Act_minus_nonact(mat, ftype, imputeMissing=1, toplot=1, key = 'pw')
mat['dat_res'] = dat_res
# save
io.savemat('groupedSpectPower_%s.mat'%(ftype), mat)

##### to SUBCOR mat data
filteridx = np.intersect1d(toSUBCOR, idx_in_metaT_1)
ftype = 'toSUBCOR'
mat = stackSpectCCEP(filteridx, cropdim=[[t1,t2],[0.5, 125]], data_type = 'pw',
                          time_sample=fti['time'], freq_sample=fti['freqs'], toplot=0)
dat_res = Act_minus_nonact(mat, ftype, imputeMissing=1, toplot=1, key = 'pw')
mat['dat_res'] = dat_res
# save
io.savemat('groupedSpectPower_%s.mat'%(ftype), mat)

Save for ITPC data:

In [ ]:
##### from SUBCOR mat data
filteridx = np.intersect1d(fromSUBCOR, idx_in_metaT_1)
ftype = 'fromSUBCOR'
mat = stackSpectCCEP(filteridx, cropdim=[[t1,t2],[0.5, 125]], data_type = 'pc',
                          time_sample=fti['time'], freq_sample=fti['freqs'], toplot=0)
dat_res = Act_minus_nonact(mat, ftype, imputeMissing=1, toplot=1, key = 'pc')
mat['dat_res'] = dat_res
# save
io.savemat('groupedSpectITPC_%s.mat'%(ftype), mat)

##### to SUBCOR mat data
filteridx = np.intersect1d(toSUBCOR, idx_in_metaT_1)
ftype = 'toSUBCOR'
mat = stackSpectCCEP(filteridx, cropdim=[[t1,t2],[0.5, 125]], data_type = 'pc',
                          time_sample=fti['time'], freq_sample=fti['freqs'], toplot=0)
dat_res = Act_minus_nonact(mat, ftype, imputeMissing=1, toplot=1, key = 'pc')
mat['dat_res'] = dat_res
# save
io.savemat('groupedSpectITPC_%s.mat'%(ftype), mat)